In [41]:
import cv2
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

## Functions

In [43]:
def analyze_samples(url):
    images_in_folder = [file for file in glob.glob(os.path.join(url, "*.png"))]
    random_images = np.random.choice(images_in_folder, 4, replace=False)
    reference_image = random_images[0]
    test_images = random_images[1:4]

    corr_array = []
    mse_array = []
    
    for image in test_images:
        base_image = cv2.imread(reference_image)
        compare_image = cv2.imread(image)

        flatten_base = base_image.flatten()
        flatten_compare = compare_image.flatten()

        

        corr_array.append(pearsonr(flatten_base, flatten_compare))
        mse_array.append(mean_squared_error(flatten_base, flatten_compare))
    return corr_array, mse_array

In [107]:
def extrair_caracteristicas(img_path, grid_size=(3, 3), limiar=127):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (300, 300))  
    
    _, binary = cv2.threshold(img, limiar, 255, cv2.THRESH_BINARY)
    
    h, w = binary.shape
    gh, gw = grid_size
    h_step, w_step = h // gh, w // gw
    
    features = []
    
    for i in range(gh):
        for j in range(gw):
            sub_img = binary[i*h_step:(i+1)*h_step, j*w_step:(j+1)*w_step]
            
            white = np.sum(sub_img == 255)
            black = np.sum(sub_img == 0)
            
            # adiciona como par (white, black) para cada região
            features.extend([white, black])
    
    return features


In [144]:
def calcular_mse(img1, img2):
    return mean_squared_error(img1, img2)

In [145]:
def calcular_pearson(img1, img2):
    return pearsonr(img1.flatten(), img2.flatten())[0]

In [157]:
def calcular_metricas(features1, features2):
    corr = pearsonr(features1, features2)[0]
    mse = mean_squared_error(features1, features2)
    return corr, mse

In [149]:
def calcular_pearson2(img1, img2):
    return pearsonr(img1, img2)

In [146]:
def carregar_imagem(path, size=(200, 200)):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)  # usa grayscale p/ simplificar
    img = cv2.resize(img, size)
    return img

## Etapa A

In [162]:
folders_path = "./Dataset//"
folders_path = Path(folders_path).resolve()
folders = [p for p in folders_path.iterdir() if p.is_dir()]  # somente pastas diretas

df_metricas = pd.DataFrame(columns=["Template_Classe", "Comparada_Classe", "Pearson", "MSE"])

IMG_SIZE = (200, 200)

# -----------------------------
# Escolher template e comparar
# -----------------------------
for template_class in folders:
    # imagens da classe template
    images_template_class = glob.glob(str(template_class / "*.jpg")) + glob.glob(str(template_class / "*.png"))
    if len(images_template_class) < 4:
        continue  # precisa de pelo menos 4 imagens na classe

    # sorteia o template
    template_path = random.choice(images_template_class)
    template_img = carregar_imagem(template_path, IMG_SIZE)

    # seleciona 3 imagens diferentes da mesma classe
    same_class_imgs = random.sample([img for img in images_template_class if img != template_path], 3)

    # --- Comparações intra-classe ---
    for img_path in same_class_imgs:
        img = carregar_imagem(img_path, IMG_SIZE)
        pearson = calcular_pearson(template_img, img)
        mse = calcular_mse(template_img, img)

        df_metricas.loc[len(df_metricas)] = [template_class.stem, template_class.stem, pearson, mse]

    # --- Comparações inter-classe ---
    other_classes = [c for c in folders if c != template_class]

    for other_class in other_classes:
        images_other_class = glob.glob(str(other_class / "*.jpg")) + glob.glob(str(other_class / "*.png"))
        if len(images_other_class) < 3:
            continue

        # sorteia 3 imagens da outra classe
        other_imgs = random.sample(images_other_class, 3)

        for img_path in other_imgs:
            img = carregar_imagem(img_path, IMG_SIZE)
            pearson = calcular_pearson(template_img, img)
            mse = calcular_mse(template_img, img)

            df_metricas.loc[len(df_metricas)] = [template_class.stem, other_class.stem, pearson, mse]

# -----------------------------
# Resultado
# -----------------------------
print(df_metricas)


   Template_Classe Comparada_Classe   Pearson       MSE
0           circle           circle  0.858235  2.026375
1           circle           circle  0.815889  2.111250
2           circle           circle  0.826872  2.257400
3           circle           square  0.814239  2.205150
4           circle           square  0.876028  2.539700
5           circle           square  0.859755  2.274925
6           circle             star  0.534191  2.285150
7           circle             star  0.586164  2.575075
8           circle             star  0.607565  2.488150
9           circle         triangle  0.589742  2.085800
10          circle         triangle  0.630490  2.419325
11          circle         triangle  0.648996  2.459450
12          square           square  0.864476  1.778025
13          square           square  0.853898  1.733375
14          square           square  0.934665  1.981025
15          square           circle  0.864580  1.526925
16          square           circle  0.873802  1

In [163]:
df = df_metricas.groupby(["Template_Classe","Comparada_Classe"]).agg({
    "Pearson": "mean",
    "MSE": "mean"
}
)

In [164]:
df

Pearson       MSE
Template_Classe Comparada_Classe                    
circle          circle            0.833665  2.131675
                square            0.850007  2.339925
                star              0.575973  2.449458
                triangle          0.623076  2.321525
square          circle            0.868837  1.626492
                square            0.884346  1.830808
                star              0.595814  1.987292
                triangle          0.684597  1.826817
star            circle            0.622200  2.086567
                square            0.625001  2.038150
                star              0.702711  2.112617
                triangle          0.624634  2.053975
triangle        circle            0.669372  2.208350
                square            0.685457  2.061292
                star              0.591126  2.422675
                triangle          0.673541  2.169183

## B Template Matching

In [165]:
folders_path = "./Dataset_Folhas/"
folders_path = Path(folders_path).resolve()
folders = [p for p in folders_path.iterdir() if p.is_dir()]  # somente pastas diretas

df_metricas = pd.DataFrame(columns=["Template_Classe", "Comparada_Classe", "Pearson", "MSE"])

IMG_SIZE = (200, 200)

# -----------------------------
# Escolher template e comparar
# -----------------------------
for template_class in folders:
    # imagens da classe template
    images_template_class = glob.glob(str(template_class / "*.jpg")) + glob.glob(str(template_class / "*.png"))
    if len(images_template_class) < 4:
        continue  # precisa de pelo menos 4 imagens na classe

    # sorteia o template
    template_path = random.choice(images_template_class)
    template_img = carregar_imagem(template_path, IMG_SIZE)

    # seleciona 3 imagens diferentes da mesma classe
    same_class_imgs = random.sample([img for img in images_template_class if img != template_path], 3)

    # --- Comparações intra-classe ---
    for img_path in same_class_imgs:
        img = carregar_imagem(img_path, IMG_SIZE)
        pearson = calcular_pearson(template_img, img)
        mse = calcular_mse(template_img, img)

        df_metricas.loc[len(df_metricas)] = [template_class.stem, template_class.stem, pearson, mse]

    # --- Comparações inter-classe ---
    other_classes = [c for c in folders if c != template_class]

    for other_class in other_classes:
        images_other_class = glob.glob(str(other_class / "*.jpg")) + glob.glob(str(other_class / "*.png"))
        if len(images_other_class) < 3:
            continue

        # sorteia 3 imagens da outra classe
        other_imgs = random.sample(images_other_class, 3)

        for img_path in other_imgs:
            img = carregar_imagem(img_path, IMG_SIZE)
            pearson = calcular_pearson(template_img, img)
            mse = calcular_mse(template_img, img)

            df_metricas.loc[len(df_metricas)] = [template_class.stem, other_class.stem, pearson, mse]

# -----------------------------
# Resultado
# -----------------------------
print(df_metricas)


    Template_Classe Comparada_Classe   Pearson       MSE
0   Acer_Capillipes  Acer_Capillipes  0.892668  1.912875
1   Acer_Capillipes  Acer_Capillipes  0.915097  1.726000
2   Acer_Capillipes  Acer_Capillipes  0.880893  1.794575
3   Acer_Capillipes        Acer_Mono  0.469517  2.624900
4   Acer_Capillipes        Acer_Mono  0.653136  1.931575
5   Acer_Capillipes        Acer_Mono  0.622529  1.998450
6   Acer_Capillipes      Acer_Opalus  0.645491  1.796375
7   Acer_Capillipes      Acer_Opalus  0.601608  1.876950
8   Acer_Capillipes      Acer_Opalus  0.638852  1.822450
9         Acer_Mono        Acer_Mono  0.765420  2.502075
10        Acer_Mono        Acer_Mono  0.825687  2.844800
11        Acer_Mono        Acer_Mono  0.792791  2.396350
12        Acer_Mono  Acer_Capillipes  0.574866  2.601300
13        Acer_Mono  Acer_Capillipes  0.593966  2.547600
14        Acer_Mono  Acer_Capillipes  0.556838  2.742325
15        Acer_Mono      Acer_Opalus  0.626348  2.536725
16        Acer_Mono      Acer_O

In [166]:
df = df_metricas.groupby(["Template_Classe","Comparada_Classe"]).agg({
    "Pearson": "mean",
    "MSE": "mean"
}
)

In [167]:
df

Pearson       MSE
Template_Classe Comparada_Classe                    
Acer_Capillipes Acer_Capillipes   0.896220  1.811150
                Acer_Mono         0.581727  2.184975
                Acer_Opalus       0.628650  1.831925
Acer_Mono       Acer_Capillipes   0.575224  2.630408
                Acer_Mono         0.794633  2.581075
                Acer_Opalus       0.622643  2.505750
Acer_Opalus     Acer_Capillipes   0.642133  1.875200
                Acer_Mono         0.664571  2.270683
                Acer_Opalus       0.824560  1.774725

## C Feature Matching

In [168]:
dataset_paths = ["Dataset/", "./Dataset_Folhas/"]

In [173]:
for folders_path in dataset_paths:
    
    folders_path = Path(folders_path).resolve()
    folders = [p for p in folders_path.rglob('*') if p.is_dir()]
    
    # DataFrame final
    df_metricas = pd.DataFrame(columns=["Classe_template", "Classe_comparada",
                                        "Imagem_template", "Imagem_comparada",
                                        "Correlação de Pearson", "MSE"])
    
    # --- Escolher uma classe como referência (exemplo: primeira pasta encontrada)
    classe_ref = folders[1]  # p[0] é o root, então pegamos p[1]
    imagens_ref = glob.glob(os.path.join(classe_ref, "*.[pj][pn]g"))
    
    # Template = primeira imagem da classe de referência
    template_path = imagens_ref[0]
    template_feat = extrair_caracteristicas(template_path)
    
    # 3 imagens da mesma classe
    same_class_imgs = np.random.choice(imagens_ref[1:], 3, replace=False)
    
    # 3 imagens de outras classes (pegando 1 de 3 classes diferentes)
    other_classes = [f for f in folders if f != classe_ref]
    other_imgs = []
    for oc in other_classes[:3]:  # pega 3 classes diferentes
        imgs = glob.glob(os.path.join(oc, "*.[pj][pn]g"))
        if len(imgs) > 0:
            other_imgs.append(np.random.choice(imgs, 1)[0])   
    
    # --- Comparações intra-classe
    for img_path in same_class_imgs:
        feat = extrair_caracteristicas(img_path)
        corr, mse = calcular_metricas(template_feat, feat)
    
        df_metricas.loc[len(df_metricas)] = [
            classe_ref.stem, classe_ref.stem,
            os.path.basename(template_path), os.path.basename(img_path),
            corr, mse
        ]
    
    # --- Comparações inter-classes
    for img_path in other_imgs:
        feat = extrair_caracteristicas(img_path)
        corr = calcular_pearson2(template_feat, feat)[0]
        mse = calcular_mse(template_feat, feat)
    
        df_metricas.loc[len(df_metricas)] = [
            classe_ref.stem, Path(img_path).parent.stem,
            os.path.basename(template_path), os.path.basename(img_path),
            corr, mse
        ]
    df = df_metricas.groupby(["Classe_template","Classe_comparada"]).agg({
    "Correlação de Pearson": "mean",
    "MSE": "mean"
    }
    )
    print(df)
    print("\n")
    print(df_metricas)
    print("\n")
    


                                  Correlação de Pearson           MSE
Classe_template Classe_comparada                                     
square          circle                         0.988336  4.778176e+05
                square                         0.994470  2.055570e+05
                star                           0.956143  1.685709e+06
                triangle                       0.949160  1.986422e+06


  Classe_template Classe_comparada Imagem_template Imagem_comparada  \
0          square           square        1166.png          461.png   
1          square           square        1166.png          847.png   
2          square           square        1166.png           57.png   
3          square           circle        1166.png          253.png   
4          square             star        1166.png         1642.png   
5          square         triangle        1166.png          616.png   

   Correlação de Pearson           MSE  
0               0.997608  1.067493e+05 

In [170]:
df_metricas

,Classe_template,Classe_comparada,Imagem_template,Imagem_comparada,Correlação de Pearson,MSE
0,Acer_Mono,Acer_Mono,216.jpg,499.jpg,0.976602,5.850838e+05
1,Acer_Mono,Acer_Mono,216.jpg,288.jpg,0.953253,1.118507e+06
2,Acer_Mono,Acer_Mono,216.jpg,807.jpg,0.972567,6.629338e+05
3,Acer_Mono,Acer_Capillipes,216.jpg,610.jpg,0.701148,6.486652e+06
4,Acer_Mono,Acer_Opalus,216.jpg,1.jpg,0.692488,6.538634e+06
